# Chapter 08 — Late Interaction, Hybrid Fusion, Reranking, Filtering

*Where we are:* combine the two retrieval arms, then sharpen precision.

```
BM25 ─┐
      ├─ RRF fusion → cross-encoder rerank → metadata filter → context
dense ┘
```

In [1]:
# === Chapter 08 · standard bootstrap (identical pattern in every notebook) ===
# Runs standalone on a fresh Google Colab VM *or* a local checkout.
import os, sys, subprocess

REPO_URL = "https://github.com/rsalehin/patent-rag-masterclass"
NEED_OCR = False
IN_COLAB = "google.colab" in sys.modules


def _clone_repo(url, target):
    """Clone the repo on Colab. For a PRIVATE repo, authenticate with a GitHub token read from
    Colab Secrets (key 'GITHUB_TOKEN') or the GITHUB_TOKEN env var. The token is never printed."""
    token = None
    try:
        from google.colab import userdata  # type: ignore
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = os.environ.get("GITHUB_TOKEN")
    auth_url = url
    if token and url.startswith("https://github.com/"):
        auth_url = url.replace("https://github.com/", f"https://{token}@github.com/")
    r = subprocess.run(["git", "clone", "--depth", "1", auth_url, target],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)  # avoid leaking the token
    if r.returncode != 0:
        raise RuntimeError(
            "git clone failed. This is a PRIVATE repo, so Colab needs a GitHub token:\n"
            "  1) Create a token (scope: repo) at https://github.com/settings/tokens\n"
            "  2) In Colab, open the key icon (Secrets) in the left sidebar, add a secret named\n"
            "     GITHUB_TOKEN, paste the token, and enable 'Notebook access'.\n"
            "  3) Re-run this cell.\n"
            "  (Alternatively, make the GitHub repo public — then no token is needed.)")


if IN_COLAB:
    target = "/content/patent-rag-masterclass"
    if not os.path.isdir(target):
        if not REPO_URL:
            raise RuntimeError("Set REPO_URL to this repo's GitHub URL (see README.md).")
        _clone_repo(REPO_URL, target)
    os.chdir(target)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    if NEED_OCR:
        subprocess.run(["apt-get", "install", "-y", "-q", "tesseract-ocr"], check=False)

# Ensure the repo root (containing patentrag/) is importable.
for _cand in [os.getcwd()] + [os.path.dirname(os.getcwd())]:
    if os.path.isdir(os.path.join(_cand, "patentrag")):
        if _cand not in sys.path:
            sys.path.insert(0, _cand)
        break

from patentrag import bootstrap as bs
bs.setup_environment(REPO_URL, need_ocr=NEED_OCR)
bs.set_seeds()
_env = bs.environment_report()
print("Chapter 08 bootstrap OK")
print("  Python", _env["python"], "| Colab:", _env["in_colab"], "| CPU cores:", _env["cpu_count"])
print("  torch", _env["torch"], "| CUDA:", _env["cuda_available"], "| tesseract:", _env["tesseract"])

Chapter 08 bootstrap OK
  Python 3.12.10 | Colab: False | CPU cores: 24
  torch 2.12.0.dev20260304+cu130 | CUDA: True | tesseract: True


## 29–30. Why one vector per document loses information — ColBERT MaxSim

A **bi-encoder** compresses a whole passage into a single vector, blurring token-level detail.
**Late interaction** (ColBERT) keeps *per-token* embeddings and scores with **MaxSim** — each
query token takes its best match against any document token:

$$\text{MaxSim}(q,d)=\sum_{i\in q}\max_{j\in d}\cos(\mathbf{q}_i,\mathbf{d}_j)$$

We implement the principle with transformer token embeddings (educational). Production ColBERT
(PLAID) adds token compression + an inverted index over centroids; we describe that rather than
install the full heavy stack.

In [2]:
from patentrag.fusion import token_embeddings, maxsim
query = "approximate nearest neighbor vector search"
passages = {
    "relevant":  "an index for approximate nearest neighbor search over high-dimensional vectors",
    "distractor":"a method for boiling vegetables to make a nearest soup",
}
qt = token_embeddings(query)
scores = {name: maxsim(qt, token_embeddings(p)) for name, p in passages.items()}
print("MaxSim (late interaction) scores:")
for k, v in scores.items():
    print(f"  {k:11}: {v:.2f}")
assert scores["relevant"] > scores["distractor"]

C:\Users\rsalehin\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13492.40it/s]

MaxSim (late interaction) scores:
  relevant   : 6.45
  distractor : 3.02


## 31. Why hybrid — BM25 and dense fail on *different* queries

We show one query where **BM25 wins** (an exact token dense retrieval smears) and one where
**dense wins** (a paraphrase BM25 misses). Neither alone is enough.

In [3]:
import pandas as pd
from patentrag.dense import DenseRetriever
chunks = bs.ensure("chunks"); by_id = {c.chunk_id: c for c in chunks}
bm25 = bs.ensure("bm25_index"); emb = bs.ensure("embeddings")
dense = DenseRetriever(emb["chunk_ids"], emb["matrix"])

def doc_rank(results, target):
    for r, (cid, _) in enumerate(results, 1):
        if by_id[cid].document_id == target:
            return r
    return None

cases = [
    ("exact token 'G06N' (BM25 should win)", "G06N", "US11971885B2"),
    ("paraphrase (dense should win)", "let shoppers get instant suggestions from taste vectors", "US11113744B2"),
]
pd.DataFrame([{"query": desc, "BM25 rank": doc_rank(bm25.search(q, 50), t),
               "Dense rank": doc_rank(dense.search(q, 50), t)} for desc, q, t in cases])

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14439.91it/s]

,query,BM25 rank,Dense rank
0,exact token 'G06N' (BM25 should win),2,32
1,paraphrase (dense should win),3,1


## 32. Reciprocal Rank Fusion

$$\text{RRF}(d)=\sum_{r\in\text{retrievers}}\frac{1}{k+\text{rank}_r(d)}\qquad(k=60)$$

RRF fuses **ranks**, not scores, so it needs no score normalization across incomparable
retrievers. We fuse BM25 + dense and show the contributing ranks.

In [4]:
from patentrag.fusion import reciprocal_rank_fusion
from patentrag.evaluation import ranked_docs_from_chunks
q = "efficient nearest neighbor search over stored feature vectors"
bm_docs = ranked_docs_from_chunks([c for c, _ in bm25.search(q, 40)], by_id)
dn_docs = ranked_docs_from_chunks([c for c, _ in dense.search(q, 40)], by_id)
fused = reciprocal_rank_fusion([bm_docs, dn_docs])
rows = []
for rank, (doc, score) in enumerate(fused[:8], 1):
    rows.append({"rank": rank, "document": doc,
                 "BM25_rank": bm_docs.index(doc)+1 if doc in bm_docs else None,
                 "dense_rank": dn_docs.index(doc)+1 if doc in dn_docs else None,
                 "RRF_score": round(score, 4)})
pd.DataFrame(rows)

,rank,document,BM25_rank,dense_rank,RRF_score
0,1,US11714853B2,1.0,1.0,0.0328
1,2,US11216459B2,3.0,3.0,0.0317
2,3,US11257279B2,2.0,NaN,0.0161
3,4,US11093561B2,NaN,2.0,0.0161
4,5,US11971885B2,NaN,4.0,0.0156


## 33–34. Cross-encoder reranking (top-50 → top-10)

A **bi-encoder** scores query and passage *independently* (fast, cacheable). A **cross-encoder**
scores them **jointly** — far more accurate, but $O(\text{candidates})$ model calls, so it only
reranks a shortlist. We retrieve 50 by hybrid fusion, then rerank with
`ms-marco-MiniLM-L-6-v2`.

In [5]:
from patentrag.fusion import CrossEncoderReranker
reranker = CrossEncoderReranker()
q2 = "choose which preview fields to show and learn from user dwell time"
fused_chunks = [by_id[cid] for cid, _ in reciprocal_rank_fusion(
    [[c for c, _ in bm25.search(q2, 50)], [c for c, _ in dense.search(q2, 50)]])[:50]]
reranked = reranker.rerank(q2, [(c.chunk_id, c.for_index()) for c in fused_chunks], top_k=50)

def show(order_ids, label, scores=None):
    return pd.DataFrame([{
        f"{label}": i + 1,
        "pub": by_id[cid].publication_number,
        "section": by_id[cid].section[:16],
        "text": by_id[cid].text[:44],
        **({"CE_score": round(scores[cid], 2)} if scores else {}),
    } for i, cid in enumerate(order_ids[:5])])

ce = {cid: s for cid, s in reranked}
before_tbl = show([c.chunk_id for c in fused_chunks], "hybrid_rank")
after_tbl = show([cid for cid, _ in reranked], "reranked_rank", ce)
print("TOP-5 BEFORE reranking (hybrid RRF order):")
print(before_tbl.to_string(index=False))
print("\nTOP-5 AFTER cross-encoder reranking (joint query-passage scoring):")
print(after_tbl.to_string(index=False))
print("\nThe cross-encoder reorders the shortlist by joint relevance; its aggregate lift")
print("(MRR 0.96 → 1.00, NDCG@10 0.96 → 0.99) is *measured* over the benchmark in Chapter 09.")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 13915.19it/s]

TOP-5 BEFORE reranking (hybrid RRF order):
 hybrid_rank          pub section                                         text
           1 US10261954B2  Claims One or more non-transitory computer-readable
           2 US10261954B2  Claims The computing system of claim 9 , wherein: t
           3 US10261954B2  Claims A computing system comprising: one or more p
           4 US10261954B2  Claims A method performed by a computing system com
           5 US10261954B2  Claims The computing system of claim 7 , wherein th

TOP-5 AFTER cross-encoder reranking (joint query-passage scoring):
 reranked_rank          pub section                                         text  CE_score
             1 US10261954B2  Claims The computing system of claim 9 , wherein: t     -0.89
             2 US10261954B2  Claims The computing system of claim 7 , wherein th     -0.96
             3 US10261954B2  Claims The method of claim 1 , further comprising:      -1.34
             4 US10261954B2  Claims The method of c

## 35–36. Metadata filtering, pre- vs post-filter

Patent search constantly filters on **biblio metadata** (jurisdiction, dates, CPC/IPC, applicant,
language). *Where* the filter runs matters:

- **Pre-filter** (`filter → ANN`): restrict the candidate universe first — exact, but you rebuild/
  restrict the index per filter and a selective filter may leave few graph neighbours.
- **Post-filter** (`ANN → filter`): retrieve then drop — simple, but a selective filter can
  **under-fill** top-k (you asked for 10, ANN returned 10, 7 survive the filter).

We use a real, selective filter: **publication year ≥ 2020**.

In [6]:
from patentrag.fusion import make_filter, post_filter, pre_filter_ids
from patentrag.sparse import BM25Retriever
docs = {d.doc_id: d for d in bs.ensure("docs_canonical")}
pred = make_filter(min_pub_year=2020)
recent = [d for d in docs.values() if d.publication_date and d.publication_date.year >= 2020]
print(f"corpus: {len(docs)} patents; matching filter (pub year ≥ 2020): {len(recent)}")

q3 = "select fragments of a document to put in the index"   # matches mostly older (pre-2020) patents
res = bm25.search(q3, 10)   # ask for 10
post = post_filter(res, by_id, docs, pred, k=10)
print(f"\nPOST-filter: retrieved top-10, only {len(post)} survive the year≥2020 filter — UNDER-FILL")

pre_ids = pre_filter_ids(chunks, docs, pred)
pre_bm25 = BM25Retriever().fit(pre_ids, [by_id[c].for_index() for c in pre_ids])
pre = pre_bm25.search(q3, 10)
print(f"PRE-filter : searched only the {len(pre_ids)} in-scope chunks → returns a full {len(pre)} results")
assert all(docs[by_id[c].document_id].publication_date.year >= 2020 for c, _ in pre)

corpus: 15 patents; matching filter (pub year ≥ 2020): 7

POST-filter: retrieved top-10, only 1 survive the year≥2020 filter — UNDER-FILL
PRE-filter : searched only the 476 in-scope chunks → returns a full 10 results


**Production implications.** RRF is the robust default fusion (cheap, calibration-free). A
cross-encoder is the single highest-leverage precision upgrade but adds latency proportional to
the shortlist — rerank 50, not 5,000. Choose pre- vs post-filter by **selectivity**: pre-filter
when the filter is very selective (avoid under-fill); post-filter when it is broad (avoid
rebuilding indexes). Production vector DBs implement filtered-ANN to get both.

## Chapter invariants

In [7]:
assert scores["relevant"] > scores["distractor"]                 # late interaction ranks correctly
assert fused[0][1] >= fused[-1][1]                               # RRF sorted
assert len(post) <= 10 and len(pre) <= 10
assert all(docs[by_id[c].document_id].publication_date.year >= 2020 for c, _ in pre)
print("All Chapter 08 invariants hold.")

All Chapter 08 invariants hold.


In [8]:
# === Chapter 08 validation footer ===
import time, platform, sys, importlib.metadata as _md
_pkgs = ['sentence-transformers', 'transformers', 'rank-bm25']
print("Chapter 08 — environment")
print("  Python :", sys.version.split()[0], "on", platform.system(), platform.release())
for _p in _pkgs:
    try: print(f"  {_p:24}: {_md.version(_p)}")
    except Exception: print(f"  {_p:24}: (not installed)")
print()
print("CHAPTER 08 VALIDATION: PASS")

Chapter 08 — environment
  Python : 3.12.10 on Windows 11
  sentence-transformers   : 6.0.0
  transformers            : 5.16.1
  rank-bm25               : 0.2.2

CHAPTER 08 VALIDATION: PASS
